<a href="https://colab.research.google.com/github/AnDinh039/PST-PRIM---RMAT_128.edges---Julia/blob/main/PST_PRIM_RMAT_128_edges_Duckdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install duckdb

In [2]:
import os
import random
import time
import tracemalloc
import duckdb

CSV_FILE = "RMAT_128_edges.csv"


# ==============================================================================
# 0. HÀM TỰ SINH FILE RMAT_128_edges.csv (NẾU TRONG THƯ MỤC CHƯA CÓ SẴN)
# ==============================================================================
def ensure_rmat_128_csv(filepath: str, num_vertices: int = 128, num_edges: int = 1280):
    if os.path.exists(filepath):
        return
    print(f"[Thông báo] Không tìm thấy '{filepath}'. Đang tạo file chuẩn RMAT-128...")
    random.seed(42)
    scale = int(round(num_vertices).bit_length() - 1)
    a, b, c, d = 0.57, 0.19, 0.19, 0.05
    edge_set = {}

    # Đảm bảo đồ thị liên thông
    for i in range(1, num_vertices):
        edge_set[(i, i + 1)] = round(random.uniform(1.0, 100.0), 2)

    while len(edge_set) < num_edges:
        u, v = 0, 0
        for depth in range(scale - 1, -1, -1):
            r = random.random()
            bit_u = 1 if r > (a + b) else 0
            bit_v = 1 if ((a < r <= a + b) or (r > a + b + c)) else 0
            u |= bit_u << depth
            v |= bit_v << depth
        u += 1
        v += 1
        if u != v and (u, v) not in edge_set and (v, u) not in edge_set:
            edge_set[(u, v)] = round(random.uniform(1.0, 100.0), 2)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("src,dst,weight\n")
        for (u, v), w in sorted(edge_set.items()):
            f.write(f"{u},{v},{w}\n")
    print(f"[Hoàn tất] Đã lưu '{filepath}' ({len(edge_set)} cạnh).\n")


# ==============================================================================
# 1. KHỞI TẠO DUCKDB & IMPORT DỮ LIỆU TỪ RMAT_128_edges.csv
# ==============================================================================
ensure_rmat_128_csv(CSV_FILE)

con = duckdb.connect(database=":memory:")

# Đọc file CSV tự động (hỗ trợ cả file có hoặc không có header, 2 hoặc 3 cột)
raw_cols = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{CSV_FILE}')").fetchall()
if len(raw_cols) >= 3:
    col_x, col_y, col_w = raw_cols[0][0], raw_cols[1][0], raw_cols[2][0]
    con.execute(f"""
        CREATE TABLE edge AS
        SELECT CAST("{col_x}" AS INTEGER) AS x,
               CAST("{col_y}" AS INTEGER) AS y,
               CAST("{col_w}" AS DOUBLE)  AS w
        FROM read_csv_auto('{CSV_FILE}')
    """)
else:
    col_x, col_y = raw_cols[0][0], raw_cols[1][0]
    con.execute(f"""
        CREATE TABLE edge AS
        SELECT CAST("{col_x}" AS INTEGER) AS x,
               CAST("{col_y}" AS INTEGER) AS y,
               1.0::DOUBLE                AS w
        FROM read_csv_auto('{CSV_FILE}')
    """)

num_edges = con.execute("SELECT COUNT(*) FROM edge").fetchone()[0]
num_nodes = con.execute("""
    SELECT COUNT(DISTINCT node) FROM (
        SELECT x AS node FROM edge UNION SELECT y AS node FROM edge
    )
""").fetchone()[0]

print("=" * 88)
print(f"THÔNG TIN DATASET TRÊN DUCKDB: {CSV_FILE}")
print(f"  - Số đỉnh (Vertices) : {num_nodes}")
print(f"  - Số cạnh (Edges)    : {num_edges} (Vô hướng uedge: {num_edges * 2})")
print("=" * 88)


# ==============================================================================
# 2. CÂU TRUY VẤN ĐỆ QUY PRIM-MST TRÊN DUCKDB (Theo Hình 8b & Mục 6.1 bài báo)
# ==============================================================================
# Lưu ý: CTE `tree_edge(t, x, y, w)` của DuckDB buộc phải nối thêm (UNION ALL)
# toàn bộ kết quả của bước t vào bảng tạm chung qua từng vòng lặp.
PRIM_CTE_CORE = """
WITH RECURSIVE
uedge(x, y, w) AS (
    SELECT x, y, w FROM edge
    UNION
    SELECT y, x, w FROM edge
),
tree_edge(t, x, y, w) AS (
    -- Bước cơ sở (t = 0): Chọn 1 cạnh có trọng số nhỏ nhất toàn đồ thị
    (
        SELECT 0 AS t, x, y, w
        FROM uedge
        ORDER BY w ASC, x ASC, y ASC
        LIMIT 1
    )
    UNION ALL
    (
        -- Bước đệ quy (t -> t + 1)
        WITH current_t AS (
            SELECT MAX(t) AS max_t FROM tree_edge
        ),
        current_edges AS (
            SELECT te.t, te.x, te.y, te.w
            FROM tree_edge te
            JOIN current_t ct ON te.t = ct.max_t
        ),
        tree_node AS (
            SELECT x AS node FROM current_edges
            UNION
            SELECT y AS node FROM current_edges
        ),
        min_edge AS (
            SELECT u.x, u.y, u.w
            FROM uedge u
            WHERE u.x IN (SELECT node FROM tree_node)
              AND u.y NOT IN (SELECT node FROM tree_node)
            ORDER BY u.w ASC, u.x ASC, u.y ASC
            LIMIT 1
        )
        -- Sao chép toàn bộ các cạnh cũ sang bước t + 1
        SELECT ce.t + 1 AS t, ce.x, ce.y, ce.w
        FROM current_edges ce
        WHERE EXISTS (SELECT 1 FROM min_edge)

        UNION ALL

        -- Cộng thêm cạnh mới vừa tìm được ở bước t + 1
        SELECT ct.max_t + 1 AS t, me.x, me.y, me.w
        FROM min_edge me
        CROSS JOIN current_t ct
    )
)
"""

# Truy vấn lấy kết quả cuối cùng: tree_edge(last, x, y)
FINAL_MST_QUERY = PRIM_CTE_CORE + """
SELECT x, y, w
FROM tree_edge
WHERE t = (SELECT MAX(t) FROM tree_edge)
ORDER BY w ASC, x ASC, y ASC;
"""


# ==============================================================================
# 3. DIỄN GIẢI TỪNG BƯỚC TÍNH TOÁN TRONG BẢNG ĐỆ QUY CỦA DUCKDB
# ==============================================================================
print("\n--- DIỄN GIẢI TỪNG BƯỚC LẶP TRONG DUCKDB (WITH RECURSIVE) ---")
con.execute(f"CREATE TEMP TABLE trace_tree_edge AS {PRIM_CTE_CORE} SELECT * FROM tree_edge;")

max_t = con.execute("SELECT MAX(t) FROM trace_tree_edge").fetchone()[0]
total_rows_stored = con.execute("SELECT COUNT(*) FROM trace_tree_edge").fetchone()[0]

# Truy vấn tìm cạnh mới được thêm vào ở mỗi bước t và tổng số dòng tích lũy trong RAM
step_details = con.execute("""
    WITH new_edge_per_step AS (
        SELECT t, x, y, w FROM trace_tree_edge WHERE t = 0
        UNION ALL
        SELECT curr.t, curr.x, curr.y, curr.w
        FROM trace_tree_edge curr
        LEFT JOIN trace_tree_edge prev
          ON prev.t = curr.t - 1 AND prev.x = curr.x AND prev.y = curr.y
        WHERE curr.t > 0 AND prev.x IS NULL
    ),
    cum_stats AS (
        SELECT t,
               COUNT(*) AS edges_in_step_t,
               SUM(COUNT(*)) OVER (ORDER BY t) AS cumulative_tuples_in_ram
        FROM trace_tree_edge
        GROUP BY t
    )
    SELECT s.t, n.x, n.y, n.w, s.edges_in_step_t, s.cumulative_tuples_in_ram
    FROM cum_stats s
    JOIN new_edge_per_step n ON s.t = n.t
    ORDER BY s.t;
""").fetchall()

for t, x, y, w, edges_in_step, cum_tuples in step_details:
    if t < 3 or t % 25 == 0 or t == max_t:
        print(
            f"  [Bước t={t:3d}] +Cạnh ({x:3d}, {y:3d}) w={w:6.2f} | "
            f"Số cạnh ở tầng t: {edges_in_step:3d} | "
            f"Tổng dòng tích lũy trong bảng tạm: {cum_tuples:5d}"
        )

con.execute("DROP TABLE trace_tree_edge;")


# ==============================================================================
# 4. ĐO THỜI GIAN THỰC THI & BỘ NHỚ TIÊU TỐN
# ==============================================================================
# Chạy warm-up 1 lần
con.execute(FINAL_MST_QUERY).fetchall()

tracemalloc.start()
t_start = time.perf_counter()

mst_edges = con.execute(FINAL_MST_QUERY).fetchall()

t_end = time.perf_counter()
current_mem, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

exec_time_ms = (t_end - t_start) * 1000.0
total_mst_weight = sum(row[2] for row in mst_edges)
# Mỗi bộ (t: INT32, x: INT32, y: INT32, w: DOUBLE) trong bảng tạm chiếm tối thiểu 20 bytes dữ liệu thô + overhead
raw_buffer_kb = (total_rows_stored * 24) / 1024.0

print("\n" + "=" * 88)
print("KẾT QUẢ ĐO LƯỜNG TRÊN DUCKDB (BỘ DỮ LIỆU RMAT_128_edges.csv)")
print("=" * 88)
print(f"  - Tổng số bước lặp (t = 0..last)         : {max_t} bước (tạo {max_t + 1} strata)")
print(f"  - Số cạnh trong Cây khung nhỏ nhất (MST) : {len(mst_edges)} cạnh")
print(f"  - Tổng trọng số MST                      : {total_mst_weight:.2f}")
print(f"  - Tổng số dòng lưu trong bảng đệ quy     : {total_rows_stored:,} dòng (bậc hai: 1 + 2 + ... + 127)")
print(f"  - Bộ nhớ dữ liệu bảng đệ quy trung gian  : ~{raw_buffer_kb:.2f} KB (Peak Python/Wrapper: {peak_mem / 1024:.2f} KB)")
print(f"  - Thời gian thực thi truy vấn DuckDB     : {exec_time_ms:.2f} ms")
print("=" * 88)

print("\n5 cạnh đầu tiên có trọng số nhỏ nhất trong MST:")
for x, y, w in mst_edges[:5]:
    print(f"  ({x:3d}, {y:3d}) -> trọng số: {w:6.2f}")





THÔNG TIN DATASET TRÊN DUCKDB: RMAT_128_edges.csv
  - Số đỉnh (Vertices) : 128
  - Số cạnh (Edges)    : 1280 (Vô hướng uedge: 2560)

--- DIỄN GIẢI TỪNG BƯỚC LẶP TRONG DUCKDB (WITH RECURSIVE) ---
  [Bước t=  0] +Cạnh ( 31,  84) w=  1.00 | Số cạnh ở tầng t:   1 | Tổng dòng tích lũy trong bảng tạm:     1
  [Bước t=  1] +Cạnh ( 84,  22) w=  7.00 | Số cạnh ở tầng t:   2 | Tổng dòng tích lũy trong bảng tạm:     3
  [Bước t=  2] +Cạnh ( 22,  45) w=  6.00 | Số cạnh ở tầng t:   3 | Tổng dòng tích lũy trong bảng tạm:     6
  [Bước t= 25] +Cạnh ( 51,  50) w=  7.00 | Số cạnh ở tầng t:  26 | Tổng dòng tích lũy trong bảng tạm:   351
  [Bước t= 50] +Cạnh ( 48,  49) w=  6.00 | Số cạnh ở tầng t:  51 | Tổng dòng tích lũy trong bảng tạm:  1326
  [Bước t= 75] +Cạnh ( 87, 104) w= 14.00 | Số cạnh ở tầng t:  76 | Tổng dòng tích lũy trong bảng tạm:  2926
  [Bước t=100] +Cạnh ( 74,  73) w= 15.00 | Số cạnh ở tầng t: 101 | Tổng dòng tích lũy trong bảng tạm:  5151
  [Bước t=125] +Cạnh ( 98,  28) w=  5.00 | Số cạn

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


KẾT QUẢ ĐO LƯỜNG TRÊN DUCKDB (BỘ DỮ LIỆU RMAT_128_edges.csv)
  - Tổng số bước lặp (t = 0..last)         : 126 bước (tạo 127 strata)
  - Số cạnh trong Cây khung nhỏ nhất (MST) : 127 cạnh
  - Tổng trọng số MST                      : 1301.00
  - Tổng số dòng lưu trong bảng đệ quy     : 8,128 dòng (bậc hai: 1 + 2 + ... + 127)
  - Bộ nhớ dữ liệu bảng đệ quy trung gian  : ~190.50 KB (Peak Python/Wrapper: 22.76 KB)
  - Thời gian thực thi truy vấn DuckDB     : 1706.12 ms

5 cạnh đầu tiên có trọng số nhỏ nhất trong MST:
  ( 31,  84) -> trọng số:   1.00
  ( 36,  37) -> trọng số:   1.00
  (  4,   3) -> trọng số:   2.00
  ( 19,  18) -> trọng số:   2.00
  ( 24,  25) -> trọng số:   2.00
